# 🏎️ FORD VEV DASHBOARD
## Google Colab — Dados da Planilha Google Sheets

In [ ]:
# ── 1. INSTALAR DEPENDÊNCIAS ──────────────────────────────
!pip install -q gspread pandas plotly folium google-auth

In [ ]:
# ── 2. AUTENTICAÇÃO GOOGLE SHEETS ─────────────────────────
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
client = gspread.authorize(creds)

# ── COLE AQUI O NOME EXATO DA SUA PLANILHA ───────────────
NOME_PLANILHA = ""  # Ex: "Ford VEV Database"

if not NOME_PLANILHA:
    NOME_PLANILHA = input("Digite o nome da planilha: ")

sheet = client.open(NOME_PLANILHA)
print(f"Conectado! Abas: {[ws.title for ws in sheet.worksheets()]}")

In [ ]:
# ── 3. FUNÇÕES DE CARGA ───────────────────────────────────
import pandas as pd

def carregar_aba(aba_nome):
    try:
        ws = sheet.worksheet(aba_nome)
        data = ws.get_all_values()
        if len(data) <= 1:
            return pd.DataFrame()
        df = pd.DataFrame(data[1:], columns=data[0])
        return df
    except:
        return pd.DataFrame()

df_turnos = carregar_aba("Turnos_Encerrados")
df_abast = carregar_aba("Abastecimentos")
df_laudos = carregar_aba("Laudos")
df_sessoes = carregar_aba("Sessoes_Teste")
df_gps = carregar_aba("GPS_Batch")
df_ativos = carregar_aba("Turnos_Ativos")

print(f"Turnos Encerrados: {len(df_turnos)} linhas")
print(f"Abastecimentos: {len(df_abast)} linhas")
print(f"Laudos: {len(df_laudos)} linhas")
print(f"Sessoes Teste: {len(df_sessoes)} linhas")
print(f"GPS Batch: {len(df_gps)} linhas")
print(f"Turnos Ativos: {len(df_ativos)} linhas")

In [ ]:
# ── 4. TRATAR DADOS ───────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
import folium
from IPython.display import display, HTML
from datetime import datetime

CORS = {
    "ford_blue": "#004899", "neon_blue": "#38bdf8",
    "green": "#10b981", "orange": "#f59e0b",
    "red": "#ef4444", "purple": "#7c3aed", "gray": "#64748b",
}

def tratar_turnos(df):
    if df.empty: return df
    for c in ["Trip KM", "Litros", "KM Rodagem", "Ciclos R389",
              "Laps Frenagem", "Total Ocorrências", "Críticas",
              "Consumo Médio (km/l)"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    if 'Data' in df.columns:
        df['Data'] = pd.to_datetime(df['Data'], dayfirst=True, errors='coerce')
    return df

df_turnos = tratar_turnos(df_turnos)

## 📊 KPIS GLOBAIS

In [ ]:
# ── 5. KPIS ───────────────────────────────────────────────
total_turnos = len(df_turnos)
total_km = df_turnos['Trip KM'].sum() if not df_turnos.empty else 0
total_litros = df_turnos['Litros'].sum() if not df_turnos.empty else 0
total_ocorrencias = df_turnos['Total Ocorrências'].sum() if not df_turnos.empty else 0
media_consumo = total_km / total_litros if total_litros > 0 else 0
total_km_rodagem = df_turnos['KM Rodagem'].sum() if not df_turnos.empty else 0
total_laudos = len(df_laudos) - 1 if not df_laudos.empty else 0

html_kpis = f"""
<div style="display:flex;gap:12px;flex-wrap:wrap;margin:20px 0;">
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #004899;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">📋 Total Turnos</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{total_turnos}</div>
  </div>
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #38bdf8;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">📏 KM Percorridos</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{total_km:,.0f} km</div>
  </div>
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #10b981;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">⛽ Litros</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{total_litros:,.1f} L</div>
  </div>
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #7c3aed;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">📊 Consumo Médio</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{media_consumo:.2f} km/l</div>
  </div>
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #ef4444;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">⚠️ Ocorrências</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{total_ocorrencias:,.0f}</div>
  </div>
  <div style="background:white;border-radius:12px;padding:16px 24px;box-shadow:0 1px 3px rgba(0,0,0,0.1);border-left:4px solid #e11d48;flex:1;min-width:140px;">
    <div style="font-size:0.75rem;color:#64748b;">📄 Laudos</div>
    <div style="font-size:1.8rem;font-weight:700;color:#0f172a;">{total_laudos}</div>
  </div>
</div>
"""
display(HTML(html_kpis))

## 📊 TURNOS

In [ ]:
# ── 6. TURNOS ─────────────────────────────────────────────
if not df_turnos.empty:
    # Turnos por dia
    turnos_dia = df_turnos.groupby(df_turnos['Data'].dt.date).size().reset_index(name='Qtd')
    turnos_dia.columns = ['Data', 'Turnos']
    fig1 = px.bar(turnos_dia, x='Data', y='Turnos', title='Turnos por Dia',
                  color_discrete_sequence=['#004899'])
    fig1.update_layout(height=350)
    fig1.show()

    # Top operadores por KM
    km_op = df_turnos.groupby('Operador')['Trip KM'].sum().sort_values(ascending=False).head(15)
    fig2 = px.bar(km_op, title='Top Operadores por KM',
                  color_discrete_sequence=['#38bdf8'],
                  labels={'value': 'KM', 'index': 'Operador'})
    fig2.update_layout(height=350, showlegend=False)
    fig2.show()

    # Ocorrencias por operador
    occ_op = df_turnos.groupby('Operador')['Total Ocorrências'].sum().sort_values(ascending=False).head(15)
    fig3 = px.bar(occ_op, title='Ocorrências por Operador',
                  color_discrete_sequence=['#ef4444'],
                  labels={'value': 'Ocorrências', 'index': 'Operador'})
    fig3.update_layout(height=350, showlegend=False)
    fig3.show()

    # Consumo histograma
    consumo = df_turnos[df_turnos['Consumo Médio (km/l)'] > 0]['Consumo Médio (km/l)']
    if not consumo.empty:
        fig4 = px.histogram(consumo, title='Distribuição Consumo (km/l)',
                            color_discrete_sequence=['#10b981'], nbins=20)
        fig4.update_layout(height=350, showlegend=False)
        fig4.show()

    display(HTML("<h3>📋 Tabela de Turnos</h3>"))
    cols = ['Data', 'Operador', 'Veiculo', 'VIN', 'Trip KM', 'Litros',
            'Consumo Medio (km/l)', 'Ciclos R389', 'Laps Frenagem',
            'Total Ocorrencias', 'Criteas', 'Posto']
    cols = [c for c in cols if c in df_turnos.columns]
    display(df_turnos[cols].sort_values('Data', ascending=False).head(50))

## ⛽ ABASTECIMENTOS

In [ ]:
# ── 7. ABASTECIMENTOS ─────────────────────────────────────
if not df_abast.empty:
    c1, c2 = st.columns(2)

    posto_count = df_abast['Posto'].value_counts().head(10)
    fig = px.pie(values=posto_count.values, names=posto_count.index,
                  title='Postos Mais Utilizados',
                  color_discrete_sequence=px.colors.qualitative.Set3)
    fig.update_layout(height=400)
    fig.show()

    df_abast['Litros'] = pd.to_numeric(df_abast['Litros'], errors='coerce').fillna(0)
    litros_posto = df_abast.groupby('Posto')['Litros'].sum().sort_values(ascending=False).head(10)
    fig = px.bar(litros_posto, title='Litros por Posto',
                  color_discrete_sequence=['#10b981'],
                  labels={'value': 'Litros', 'index': 'Posto'})
    fig.update_layout(height=400, showlegend=False)
    fig.show()

    if 'Tipo Combustivel' in df_abast.columns:
        tipo = df_abast['Tipo Combustivel'].value_counts()
        if len(tipo) > 1:
            fig = px.pie(values=tipo.values, names=tipo.index,
                          title='Tipo de Combustvel',
                          color_discrete_sequence=['#004899', '#10b981'])
            fig.update_layout(height=350)
            fig.show()

## ⚠️ LAUDOS

In [ ]:
# ── 8. LAUDOS ─────────────────────────────────────────────
if not df_laudos.empty:
    sev = df_laudos['Severidade'].value_counts()
    fig = px.pie(values=sev.values, names=sev.index, title='Por Severidade',
                  color_discrete_map={
                      'Crtico': '#ef4444', 'critico': '#ef4444',
                      'Moderado': '#f59e0b', 'moderado': '#f59e0b',
                      'Leve': '#10b981', 'leve': '#10b981',
                  })
    fig.update_layout(height=400)
    fig.show()

    if 'Categoria' in df_laudos.columns:
        cat = df_laudos['Categoria'].value_counts().head(10)
        fig = px.bar(cat, title='Top Categorias',
                      color_discrete_sequence=['#7c3aed'],
                      labels={'value': 'Qtd', 'index': 'Categoria'})
        fig.update_layout(height=350, showlegend=False)
        fig.show()

    if 'Operador' in df_laudos.columns:
        op = df_laudos['Operador'].value_counts().head(10)
        fig = px.bar(op, title='Laudos por Operador',
                      color_discrete_sequence=['#ef4444'],
                      labels={'value': 'Laudos', 'index': 'Operador'})
        fig.update_layout(height=350, showlegend=False)
        fig.show()

    display(HTML("<h3>📋 Tabela de Laudos</h3>"))
    display(df_laudos.head(100))

## 🛞 PERFORMANCE

In [ ]:
# ── 9. PERFORMANCE ────────────────────────────────────────
if not df_turnos.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_turnos['Data'], y=df_turnos['Ciclos R389'],
        mode='markers', name='Ciclos R389',
        marker=dict(color='#004899', size=6, opacity=0.6)))
    fig.add_trace(go.Scatter(
        x=df_turnos['Data'], y=df_turnos['Laps Frenagem'],
        mode='markers', name='Laps Frenagem',
        marker=dict(color='#ef4444', size=6, opacity=0.6)))
    fig.update_layout(title='Ciclos R389 vs Frenagens por Turno', height=400)
    fig.show()

    metricas = df_turnos[['Ciclos R389', 'Laps Frenagem',
                           'Laps Desaceleracao', 'KM Rodagem']].sum()
    fig = px.bar(x=metricas.index, y=metricas.values,
                  title='Total de Metricas Agregadas',
                  color_discrete_sequence=['#7c3aed'],
                  labels={'x': '', 'y': 'Total'})
    fig.update_layout(height=400, showlegend=False)
    fig.show()

## 🗺️ MAPA GPS

In [ ]:
# ── 10. GPS MAPA ──────────────────────────────────────────
if not df_gps.empty:
    df_gps['Latitude'] = pd.to_numeric(df_gps['Latitude'], errors='coerce')
    df_gps['Longitude'] = pd.to_numeric(df_gps['Longitude'], errors='coerce')
    df_gps['Velocidade (km/h)'] = pd.to_numeric(df_gps['Velocidade (km/h)'], errors='coerce').fillna(0)
    gps = df_gps.dropna(subset=['Latitude', 'Longitude'])

    print(f"Total pontos: {len(gps)} | Vel média: {gps['Velocidade (km/h)'].mean():.1f} km/h | Vel máx: {gps['Velocidade (km/h)'].max():.1f} km/h")

    centro = [gps['Latitude'].mean(), gps['Longitude'].mean()]
    m = folium.Map(location=centro, zoom_start=14, tiles='CartoDB dark')

    for _, row in gps.iterrows():
        v = row['Velocidade (km/h)']
        if v < 60: cor = '#10b981'
        elif v < 120: cor = '#f59e0b'
        else: cor = '#ef4444'
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=3, color=cor, fill=True, fillOpacity=0.7,
            popup=f"{v:.0f} km/h | {row.get('Operador', '')}",
        ).add_to(m)

    display(m)
else:
    print("Sem dados de GPS.")